## Loading the data

In [5]:
import numpy as np
from parameters import get_parameters, get_slider_params, calculate_derived_parameters
from model_run import run_model_dash
from global_func import reset_flags, reset_E, reset_HSS, reset_S
import pandas as pd
import os

In [9]:
#initialize global parameters
total_runs = 400 #200
seeds = np.random.default_rng(2025).integers(low=0, high=1e6, size=total_runs)
MODEL = {
    "int_period": 12 * 3, #three years of implementation
    "n_months": 12 * 6,   #additional three years of maintainance
}
slider_params = get_slider_params()
base_params = get_parameters()

In [10]:
def generate_baseline_df(total_runs, seeds, MODEL, ind_dir, fac_dir):
    # df_ind_list = []
    # df_fac_list = []
    n_months = MODEL["n_months"]
    int_period = MODEL["int_period"]
    b_flags = reset_flags()
    b_HSS = reset_HSS(slider_params)
    b_S = reset_S(slider_params)
    b_E = reset_E()

    for run in range(total_runs):
        os.makedirs(ind_dir, exist_ok=True)
        os.makedirs(fac_dir, exist_ok=True)

        base_seed = seeds[run]
        rng_param = np.random.default_rng(base_seed)
        b_param = get_parameters(rng = rng_param)
        b_param = calculate_derived_parameters(b_param)
        b_param.update({"E": b_E, "S": b_S, "HSS": b_HSS
        })
        _, b_outcomes, b_fac = run_model_dash(b_param, b_flags, n_months, int_period, base_seed=base_seed)
        b_fac["Run"] = run
        b_outcomes["Run"] = run
        b_outcomes["Scenario"] = "Baseline"
        b_fac["Scenario"] = "Baseline"
        #df_fac_list.append(pd.DataFrame(b_fac))
        #df_ind_list.append(pd.DataFrame(b_outcomes))
        # Write to CSV per run
        b_outcomes.to_csv(os.path.join(ind_dir, f"ind_base_run_{run}.csv"), index=False)
        b_fac.to_csv(os.path.join(fac_dir, f"fac_base_run_{run}.csv"), index=False)
    # df_ind = pd.concat(df_ind_list, ignore_index=True)
    # df_ind["Scenario"] = "Baseline"
    # df_fac = pd.concat(df_fac_list, ignore_index=True)
    # df_fac["Scenario"] = "Baseline"
    # return df_fac, df_ind

## Running the baseline model

In [11]:
ind_dir = "/Users/tingtingji/Library/CloudStorage/OneDrive-JohnsHopkins/Kakamega SDR Project/simulation results/20250920_HSS/Individual_Data/"
fac_dir = "/Users/tingtingji/Library/CloudStorage/OneDrive-JohnsHopkins/Kakamega SDR Project/simulation results/20250920_HSS/Facility_Data/"
generate_baseline_df(total_runs, seeds, MODEL, ind_dir, fac_dir)

/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)
/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)
/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)


In [12]:
def HSS_scenario(scenario_name):
    i_flags = reset_flags()
    i_HSS = reset_HSS(slider_params)
    i_S = reset_S(slider_params)
    i_E = reset_E()
    #demand
    i_flags['flag_SDR'] = 1
    i_flags['flag_CHV'] = 1
    i_flags['flag_ANC'] = 1
    i_flags['flag_LB'] = 1
    i_HSS["P_ANC"] = scenario_dic[scenario_name][0]
    i_HSS["P_L45"] = scenario_dic[scenario_name][1]
    i_HSS['CHV_memory'] = memory_dic[scenario_name]
    i_HSS['tau_decay'] = 6
    #supply
    i_flags['flag_performance'] = 1
    i_HSS["knowledge"] = scenario_dic[scenario_name][3]
    i_flags['flag_capacity'] = 1
    i_HSS["capacity_added"] = scenario_dic[scenario_name][2]
    i_flags['flag_labor'] = 1
    i_HSS["labor_ratio"] = scenario_dic[scenario_name][3]
    i_flags['flag_equipment'] = 1
    i_HSS["sensor_ratio"] = scenario_dic[scenario_name][3]
    i_flags['flag_refer'] = 1
    i_HSS["P_refer"] = scenario_dic[scenario_name][3]
    i_flags['flag_transfer'] = 1
    i_HSS["P_transfer"] = scenario_dic[scenario_name][3]
    return i_E, i_S, i_HSS, i_flags

#HSS conservative match
scenario_dic = {
    'conservative match': np.array([0.7, 0.53, 0.25, 1]),
    'conservative dismatch': np.array([0.7, 0.53, 0.125, 0.5]),
    'moderate match': np.array([0.8, 0.68, 0.5, 1]),
    'moderate dismatch': np.array([0.8, 0.68, 0.25, 0.5]),
    'aggressive match': np.array([0.9, 0.9, 0.85, 1]),
    'aggressive dismatch': np.array([0.9, 0.9, 0.425, 0.5]),
}

memory_dic = {
    'conservative match': "Always Forget",
    'conservative dismatch': "Always Forget",
    'moderate match': "Always Forget",
    'moderate dismatch': "Always Forget",
    'aggressive match': "Always Forget",
    'aggressive dismatch': "Always Forget",
}

In [13]:
def generate_intervention_df(total_runs, seeds, MODEL, scenario_name, ind_dir, fac_dir):
    # df_ind = pd.DataFrame()
    # df_ind_list = []
    # df_fac = pd.DataFrame()
    # df_fac_list = []
    n_months = MODEL["n_months"]
    int_period = MODEL["int_period"]
    i_E, i_S, i_HSS, i_flags = HSS_scenario(scenario_name)

    for run in range(total_runs):
        base_seed = seeds[run]
        rng_param = np.random.default_rng(base_seed)
        i_param = get_parameters(rng = rng_param)
        i_param = calculate_derived_parameters(i_param)
        i_param.update({"E": i_E, "S": i_S, "HSS": i_HSS
    })
        _, i_outcomes, i_fac = run_model_dash(i_param, i_flags, n_months, int_period, base_seed=base_seed)
        i_fac["Run"] = run
        i_outcomes["Run"] = run
        i_outcomes["Scenario"] = scenario_name
        i_fac["Scenario"] = scenario_name

        # Write to CSV per run
        i_outcomes.to_csv(os.path.join(ind_dir, f"ind_{scenario_name}_run_{run}.csv"), index=False)
        i_fac.to_csv(os.path.join(fac_dir, f"fac_{scenario_name}_run_{run}.csv"), index=False)
        # df_ind_list.append(pd.DataFrame(i_outcomes))
        # df_fac_list.append(pd.DataFrame(i_fac))

    # df_ind = pd.concat(df_ind_list, ignore_index=True)
    # df_ind["Scenario"] = scenario_name
    # df_fac = pd.concat(df_fac_list, ignore_index=True)
    # df_fac["Scenario"] = scenario_name
    # return df_fac, df_ind

## Running the model for different scenarios

In [14]:
generate_intervention_df(total_runs, seeds, MODEL, 'conservative match', ind_dir, fac_dir)

/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)
/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)
/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)


In [15]:
generate_intervention_df(total_runs, seeds, MODEL, 'conservative dismatch', ind_dir, fac_dir)

/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)
/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)
/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)


In [16]:
generate_intervention_df(total_runs, seeds, MODEL, 'moderate match', ind_dir, fac_dir)


/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)
/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)
/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)


In [17]:
generate_intervention_df(total_runs, seeds, MODEL, 'moderate dismatch', ind_dir, fac_dir)


/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)
/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)
/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)


In [18]:
generate_intervention_df(total_runs, seeds, MODEL, 'aggressive match', ind_dir, fac_dir)


/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)
/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)
/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)


In [19]:
generate_intervention_df(total_runs, seeds, MODEL, 'aggressive dismatch', ind_dir, fac_dir)


/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)
/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)
/Users/tingtingji/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_minpack_py.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)
